# VisionGuard evidence notebook

Run this notebook in Colab after selecting a GPU runtime. It generates the missing evaluation figures, error-analysis grid, and an eight-trial Keras Tuner result archive. It does not change the trained champion model.

In [ ]:
!git clone https://github.com/abdul-raheem-fast/23L-0588_A2_Ops.git
%cd 23L-0588_A2_Ops
%pip install -q tensorflow-cpu==2.21.0 keras-tuner==1.4.7 kagglehub scikit-learn pandas matplotlib seaborn
import os, random, shutil, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
SEED = 23
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
dataset_root = Path('/content/dataset')
dataset_root.mkdir(exist_ok=True)
if not any(dataset_root.rglob('*')):
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('cashbowman/ai-generated-images-vs-real-images'))
    print('Dataset:', downloaded)
    for item in downloaded.iterdir():
        target = dataset_root / item.name
        if item.is_dir(): shutil.copytree(item, target, dirs_exist_ok=True)
        else: shutil.copy2(item, target)
from sklearn.model_selection import train_test_split
classes = {'AiArtData': 1, 'RealArt': 0}
files = [(str(p), classes[p.parent.name]) for p in dataset_root.rglob('*') if p.is_file() and p.parent.name in classes]
frame = pd.DataFrame(files, columns=['path','label'])
train, test = train_test_split(frame, test_size=0.1, random_state=SEED, stratify=frame.label)
train, validation = train_test_split(train, test_size=1/9, random_state=SEED, stratify=train.label)
train = train.reset_index(drop=True); validation = validation.reset_index(drop=True); test = test.reset_index(drop=True)
print('Images:', len(frame), 'train:', len(train), 'validation:', len(validation), 'test:', len(test))

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
IMAGE_SIZE = (224, 224)
def load_arrays(split):
    x = np.stack([img_to_array(load_img(p, target_size=IMAGE_SIZE, color_mode='rgb')) / 255.0 for p in split.path])
    return x.astype('float32'), split.label.to_numpy().astype('int32')
x_test, y_test = load_arrays(test)
model_url = 'https://huggingface.co/spaces/ghaurixd/visionguard-inference/resolve/main/models/champion.keras'
model_path = Path('/content/champion.keras')
if not model_path.exists(): urllib.request.urlretrieve(model_url, model_path)
model = tf.keras.models.load_model(model_path)
probabilities = model.predict(x_test, verbose=0).reshape(-1)
predictions = (probabilities >= 0.5).astype('int32')
print('Champion loaded:', model_path.stat().st_size, 'bytes')
print('Test predictions:', len(predictions))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, auc, classification_report
out = Path('/content/evidence'); out.mkdir(exist_ok=True)
cm = confusion_matrix(y_test, predictions)
fig, ax = plt.subplots(figsize=(5, 4)); ConfusionMatrixDisplay(cm, display_labels=['Real','AI-generated']).plot(ax=ax, cmap='Blues', colorbar=False); ax.set_title('VisionGuard champion confusion matrix'); fig.tight_layout(); fig.savefig(out/'confusion_matrix.png', dpi=180); plt.show()
fpr, tpr, _ = roc_curve(y_test, probabilities); score = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(5, 4)); ax.plot(fpr, tpr, label=f'ROC-AUC = {score:.3f}'); ax.plot([0,1],[0,1],'--',color='gray'); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.set_title('VisionGuard champion ROC curve'); ax.legend(); fig.tight_layout(); fig.savefig(out/'roc_curve.png', dpi=180); plt.show()
print(classification_report(y_test, predictions, target_names=['Real','AI-generated'], zero_division=0))

In [ ]:
wrong = np.where(predictions != y_test)[0][:10]
fig, axes = plt.subplots(2, 5, figsize=(18, 7)); axes = axes.ravel()
for ax, idx in zip(axes, wrong):
    ax.imshow(x_test[idx]); ax.axis('off')
    true_name = 'AI-generated' if y_test[idx] else 'Real'; pred_name = 'AI-generated' if predictions[idx] else 'Real'
    ax.set_title(f'True: {true_name}\nPred: {pred_name} ({max(probabilities[idx], 1-probabilities[idx]):.2f})', fontsize=9)
fig.suptitle('VisionGuard champion error analysis', fontsize=16); fig.tight_layout(); fig.savefig(out/'error_analysis_grid.png', dpi=180); plt.show()
pd.DataFrame({'path': test.path.iloc[wrong].to_list(), 'true_label': y_test[wrong], 'predicted_label': predictions[wrong], 'probability_ai': probabilities[wrong]}).to_csv(out/'error_analysis.csv', index=False)

In [ ]:
import keras_tuner as kt
x_tune, y_tune = load_arrays(train.sample(n=min(256, len(train)), random_state=SEED))
def build_tuner_model(hp):
    inputs = tf.keras.Input((224,224,3)); x = inputs
    x = tf.keras.layers.Conv2D(hp.Int('filters', 16, 64, step=16), 3, activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x); x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1))(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    m = tf.keras.Model(inputs, outputs); m.compile(optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate',[1e-3,3e-4,1e-4])), loss='binary_crossentropy', metrics=['accuracy'])
    return m
tuner = kt.RandomSearch(build_tuner_model, objective='val_accuracy', max_trials=8, directory='/content/tuner', project_name='visionguard', overwrite=True)
tuner.search(x_tune, y_tune, validation_split=0.2, epochs=3, batch_size=32, verbose=1)
print('Completed trials:', len(tuner.oracle.trials)); print('Best hyperparameters:', tuner.get_best_hyperparameters(1)[0].values)
tuner.results_summary(num_trials=8)
shutil.make_archive('/content/visionguard-evidence', 'zip', '/content/evidence')
from google.colab import files
files.download('/content/visionguard-evidence.zip')